# Computational Cost
total parameters
trainable parameters
DINO parameters
AgriCLIP aligner parameters
SAM2 parameters
DINO GFLOPs estimate
AgriCLIP aligner GFLOPs
SAM2 GFLOPs estimate
full-pipeline GFLOPs estimate
GPU peak memory
mean/median/std latency
images/sec
total runtime for 755 images
separate DINO+AgriCLIP, SAM2, and full-pipeline benchmarks

In [42]:
# ============================================================
# CELL 1 — IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms

from datasets import load_dataset

from transformers import (
    Sam2Model,
    Sam2Processor
)

print("=" * 70)
print("COMPUTATIONAL COST ANALYSIS")
print("AgriCLIP + SAM2")
print("=" * 70)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )

print("=" * 70)

COMPUTATIONAL COST ANALYSIS
AgriCLIP + SAM2
Python: 3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]
PyTorch: 2.12.0+cu130
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 4090
CUDA: 13.0


In [43]:
# ============================================================
# CELL 2 — BENCHMARK CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Benchmark settings
# ------------------------------------------------------------

NUM_VALIDATION_IMAGES = 755

WARMUP_IMAGES = 10

# Full runtime benchmark
BENCHMARK_IMAGES = 755

# GFLOPs profiling uses one representative image
PROFILE_INDEX = 0

# AgriCLIP
TOP_K = 32

# Current box margin used in your final pipeline
BOX_MARGIN = 25

# Automatic SAM2 grid used in your previous experiment
AUTO_GRID_SIZE = 4

print("=" * 70)
print("BENCHMARK SETTINGS")
print("=" * 70)

print(
    "Validation images:",
    NUM_VALIDATION_IMAGES
)

print(
    "Warm-up images:",
    WARMUP_IMAGES
)

print(
    "Measured images:",
    BENCHMARK_IMAGES
)

print(
    "AgriCLIP K:",
    TOP_K
)

print(
    "Box margin:",
    BOX_MARGIN
)

print(
    "Automatic SAM2 grid:",
    f"{AUTO_GRID_SIZE} x {AUTO_GRID_SIZE}"
)

print("=" * 70)

BENCHMARK SETTINGS
Validation images: 755
Warm-up images: 10
Measured images: 755
AgriCLIP K: 32
Box margin: 25
Automatic SAM2 grid: 4 x 4


In [44]:
# ============================================================
# CELL 3 — PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/zahranadeem/thesis"
)

AGRICLIP_ROOT = (
    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
)

AGRICLIP_ALIGNMENT_DIR = (
    AGRICLIP_ROOT
    / "AgriCLIP_alignment"
)

# ------------------------------------------------------------
# DINO checkpoint
# ------------------------------------------------------------

DINO_CANDIDATES = [

    PROJECT_ROOT
    / "agriclip_Zahra"
    / "weights"
    / "DINO-PT.pth",

    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
    / "Weights"
    / "DINO-PT.pth",

    PROJECT_ROOT
    / "DINO-PT.pth"
]

DINO_PATH = None

for candidate in DINO_CANDIDATES:

    if candidate.exists():

        DINO_PATH = candidate

        break

# ------------------------------------------------------------
# VegAnn aligner
# ------------------------------------------------------------

ALIGNER_PATH = (
    PROJECT_ROOT
    / "vegann_aligner.pth"
)

# ------------------------------------------------------------
# ORIGINAL non-augmented SAM2 checkpoint
# ------------------------------------------------------------

SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "sam2_vegann_checkpoints"
    / "best_sam2_vegann_mask_decoder.pt"
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "computational_cost"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# SAM2 model name
# ------------------------------------------------------------

MODEL_NAME = (
    "facebook/sam2.1-hiera-tiny"
)

print("=" * 70)
print("PATH REGISTRY")
print("=" * 70)

print(
    "Project:",
    PROJECT_ROOT
)

print(
    "AgriCLIP:",
    AGRICLIP_ROOT
)

print(
    "DINO:",
    DINO_PATH
)

print(
    "Aligner:",
    ALIGNER_PATH
)

print(
    "SAM2:",
    SAM2_CHECKPOINT
)

print(
    "Results:",
    RESULTS_DIR
)

print("\nExistence checks:")

print(
    "DINO:",
    DINO_PATH is not None
    and DINO_PATH.exists()
)

print(
    "Aligner:",
    ALIGNER_PATH.exists()
)

print(
    "SAM2:",
    SAM2_CHECKPOINT.exists()
)

if DINO_PATH is None:
    raise FileNotFoundError(
        "Could not find DINO-PT.pth"
    )

assert ALIGNER_PATH.exists()
assert SAM2_CHECKPOINT.exists()

print("\n✓ All required paths exist")

PATH REGISTRY
Project: /home/zahranadeem/thesis
AgriCLIP: /home/zahranadeem/thesis/agriclip_sam2/AgriCLIP
DINO: /home/zahranadeem/thesis/agriclip_Zahra/weights/DINO-PT.pth
Aligner: /home/zahranadeem/thesis/vegann_aligner.pth
SAM2: /home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt
Results: /home/zahranadeem/thesis/results/computational_cost

Existence checks:
DINO: True
Aligner: True
SAM2: True

✓ All required paths exist


In [45]:
# ============================================================
# CELL 4 — LOAD VEGANN
# ============================================================

print("=" * 70)
print("LOADING VEGANN")
print("=" * 70)

vegann = load_dataset(
    "simonMadec/VegAnn"
)

print(
    vegann
)

print(
    "\nOriginal train size:",
    len(vegann["train"])
)

print("=" * 70)

LOADING VEGANN


DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5'],
        num_rows: 3775
    })
})

Original train size: 3775


In [46]:
# ============================================================
# CELL 5 — RECREATE EXACT VEGANN SPLIT
# ============================================================

vegann_split = (
    vegann["train"]
    .train_test_split(
        test_size=0.20,
        seed=42
    )
)

train_hf = (
    vegann_split["train"]
)

val_hf = (
    vegann_split["test"]
)

print("=" * 70)
print("VEGANN SPLIT")
print("=" * 70)

print(
    "Training:",
    len(train_hf)
)

print(
    "Validation:",
    len(val_hf)
)

assert len(val_hf) == 755

print(
    "\n✓ Validation split = 755 images"
)

print("=" * 70)

VEGANN SPLIT
Training: 3020
Validation: 755

✓ Validation split = 755 images


In [47]:
# ============================================================
# CELL 6 — DATASET COLUMN DETECTION
# ============================================================

columns = train_hf.column_names

print("=" * 70)
print("VEGANN COLUMNS")
print("=" * 70)

print(
    "Available:",
    columns
)

IMAGE_KEY = None
MASK_KEY = None

for key in [
    "image",
    "images",
    "img"
]:

    if key in columns:

        IMAGE_KEY = key

        break

for key in [
    "mask",
    "masks",
    "label",
    "labels",
    "segmentation",
    "annotation"
]:

    if key in columns:

        MASK_KEY = key

        break

print(
    "\nIMAGE_KEY:",
    IMAGE_KEY
)

print(
    "MASK_KEY:",
    MASK_KEY
)

if IMAGE_KEY is None:
    raise RuntimeError(
        "Could not find image column."
    )

if MASK_KEY is None:
    raise RuntimeError(
        "Could not find mask column."
    )

print(
    "\n✓ Dataset columns identified"
)

VEGANN COLUMNS
Available: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5']

IMAGE_KEY: image
MASK_KEY: mask

✓ Dataset columns identified


In [48]:
# ============================================================
# CELL 7 — IMAGE + MASK HELPERS
# ============================================================

def get_image(
    index,
    split=val_hf
):

    image = split[index][IMAGE_KEY]

    if isinstance(
        image,
        Image.Image
    ):

        return image.convert("RGB")

    image = np.asarray(
        image
    )

    return (
        Image.fromarray(
            image
        )
        .convert("RGB")
    )


def get_mask(
    index,
    split=val_hf
):

    mask = split[index][MASK_KEY]

    if isinstance(
        mask,
        Image.Image
    ):

        mask = np.asarray(
            mask
        )

    elif torch.is_tensor(mask):

        mask = (
            mask
            .detach()
            .cpu()
            .numpy()
        )

    else:

        mask = np.asarray(
            mask
        )

    if mask.ndim == 3:

        mask = mask[..., 0]

    return (
        mask > 0
    ).astype(np.uint8)


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_image = get_image(0)
test_mask = get_mask(0)

print("=" * 70)
print("DATA TEST")
print("=" * 70)

print(
    "Image:",
    type(test_image),
    test_image.size
)

print(
    "Mask:",
    type(test_mask),
    test_mask.shape
)

print(
    "Foreground pixels:",
    test_mask.sum()
)

print("=" * 70)

DATA TEST
Image: <class 'PIL.Image.Image'> (512, 512)
Mask: <class 'numpy.ndarray'> (512, 512)
Foreground pixels: 208628


In [49]:
# ============================================================
# CELL 8 — IMPORT AGRICLIP TEXTTOCONCEPT
# ============================================================

if str(
    AGRICLIP_ALIGNMENT_DIR
) not in sys.path:

    sys.path.insert(
        0,
        str(AGRICLIP_ALIGNMENT_DIR)
    )

from TextToConcept import (
    TextToConcept,
    ClipZeroShot
)

print("=" * 70)
print("AGRICLIP IMPORT")
print("=" * 70)

print(
    "TextToConcept:",
    TextToConcept
)

print(
    "ClipZeroShot:",
    ClipZeroShot
)

print("=" * 70)

AGRICLIP IMPORT
TextToConcept: <class 'TextToConcept.TextToConcept'>
ClipZeroShot: <class 'TextToConcept.ClipZeroShot'>


In [50]:
# ============================================================
# CELL 9 — LOAD DINO RESNET-50
# ============================================================

print("=" * 70)
print("LOADING DINO")
print("=" * 70)

print(
    "DINO checkpoint:",
    DINO_PATH
)

# ------------------------------------------------------------
# Official AgriCLIP DINO backbone
# ------------------------------------------------------------

dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_resnet50",
    pretrained=True
)

print(
    "\nDINO type:",
    type(dino_model)
)

# ------------------------------------------------------------
# Load DINO-PT.pth
# ------------------------------------------------------------

dino_checkpoint = torch.load(
    DINO_PATH,
    map_location="cpu",
    weights_only=False
)

print(
    "\nCheckpoint type:",
    type(dino_checkpoint)
)

if isinstance(
    dino_checkpoint,
    dict
):

    print(
        "Checkpoint keys:",
        list(
            dino_checkpoint.keys()
        )[:20]
    )

    if "state_dict" in dino_checkpoint:

        dino_state_dict = (
            dino_checkpoint[
                "state_dict"
            ]
        )

    elif "model_state_dict" in dino_checkpoint:

        dino_state_dict = (
            dino_checkpoint[
                "model_state_dict"
            ]
        )

    elif "model" in dino_checkpoint:

        dino_state_dict = (
            dino_checkpoint[
                "model"
            ]
        )

    else:

        dino_state_dict = (
            dino_checkpoint
        )

else:

    raise TypeError(
        "Unsupported DINO checkpoint format."
    )

# ------------------------------------------------------------
# Remove common module prefix
# ------------------------------------------------------------

clean_state_dict = {}

for key, value in (
    dino_state_dict.items()
):

    if key.startswith(
        "module."
    ):

        key = key[
            len("module.") :
        ]

    clean_state_dict[
        key
    ] = value

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

load_result = (
    dino_model.load_state_dict(
        clean_state_dict,
        strict=True
    )
)

print(
    "\nDINO load result:",
    load_result
)

dino_model = (
    dino_model
    .to(device)
)

dino_model.eval()

for parameter in (
    dino_model.parameters()
):

    parameter.requires_grad = False

print(
    "\n✓ DINO loaded and frozen"
)

print(
    "Layer4:",
    dino_model.layer4
)

print("=" * 70)

LOADING DINO
DINO checkpoint: /home/zahranadeem/thesis/agriclip_Zahra/weights/DINO-PT.pth

DINO type: <class 'torchvision.models.resnet.ResNet'>

Checkpoint type: <class 'dict'>
Checkpoint keys: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.0.conv3.weight', 'layer1.0.bn3.weight']


Using cache found in /home/zahranadeem/.cache/torch/hub/facebookresearch_dino_main
/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



DINO load result: <All keys matched successfully>

✓ DINO loaded and frozen
Layer4: Sequential(
  (0): Bottleneck(
    (conv1): Conv2d(1024, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv3): Conv2d(512, 2048, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn3): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(1024, 2048, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (1): Bottleneck(
    (conv1): Conv2d(2048, 512, kernel_size=(1, 1), str

In [51]:
# ============================================================
# CELL 10 — DINO PREPROCESSING
# ============================================================

dino_transform = transforms.Compose([

    transforms.Resize(224),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print("=" * 70)
print("DINO TRANSFORM")
print("=" * 70)

print(
    dino_transform
)

print("=" * 70)

DINO TRANSFORM
Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [52]:
# ============================================================
# CELL 11 — DINO LAYER4 HOOK
# ============================================================

captured_feature_map = None


def capture_layer4_output(
    module,
    inputs,
    output
):

    global captured_feature_map

    captured_feature_map = (
        output.detach()
    )


hook_handle = (
    dino_model
    .layer4
    .register_forward_hook(
        capture_layer4_output
    )
)

print("=" * 70)
print("DINO LAYER4 HOOK")
print("=" * 70)

print(
    "✓ Hook registered"
)

print("=" * 70)

DINO LAYER4 HOOK
✓ Hook registered


In [53]:
# ============================================================
# CELL 12 — DINO SPATIAL FEATURES
# ============================================================

@torch.no_grad()
def extract_dino_patch_features(
    pil_image
):

    global captured_feature_map

    captured_feature_map = None

    image_tensor = (
        dino_transform(
            pil_image
        )
        .unsqueeze(0)
        .to(device)
    )

    _ = dino_model(
        image_tensor
    )

    if captured_feature_map is None:

        raise RuntimeError(
            "DINO layer4 hook did not capture output."
        )

    if captured_feature_map.ndim != 4:

        raise RuntimeError(
            "Expected [B,C,H,W], got "
            f"{captured_feature_map.shape}"
        )

    _, channels, grid_h, grid_w = (
        captured_feature_map.shape
    )

    patch_features = (
        captured_feature_map[0]
        .permute(
            1,
            2,
            0
        )
        .reshape(
            grid_h * grid_w,
            channels
        )
        .contiguous()
    )

    return (
        patch_features,
        grid_h,
        grid_w
    )


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_dino_features, test_grid_h, test_grid_w = (
    extract_dino_patch_features(
        test_image
    )
)

print("=" * 70)
print("DINO FEATURE TEST")
print("=" * 70)

print(
    "Raw feature map:",
    captured_feature_map.shape
)

print(
    "Grid:",
    f"{test_grid_h} x {test_grid_w}"
)

print(
    "Patch features:",
    test_dino_features.shape
)

assert test_dino_features.shape == (
    49,
    2048
)

print(
    "\n✓ Expected 49 x 2048 features obtained"
)

print("=" * 70)

DINO FEATURE TEST
Raw feature map: torch.Size([1, 2048, 7, 7])
Grid: 7 x 7
Patch features: torch.Size([49, 2048])

✓ Expected 49 x 2048 features obtained


In [54]:
# ============================================================
# CELL 13 — LOAD AGRICLIP ALIGNER
# ============================================================

print("=" * 70)
print("LOADING VEGANN AGRICLIP ALIGNER")
print("=" * 70)

aligner_checkpoint = torch.load(
    ALIGNER_PATH,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:",
    list(
        aligner_checkpoint.keys()
    )
)

W = torch.as_tensor(
    aligner_checkpoint["W"],
    dtype=torch.float32
)

b = torch.as_tensor(
    aligner_checkpoint["b"],
    dtype=torch.float32
)

print(
    "W:",
    W.shape
)

print(
    "b:",
    b.shape
)

# ------------------------------------------------------------
# Make sure orientation is correct
# ------------------------------------------------------------

if W.shape == (
    2048,
    512
):

    W = W.T

elif W.shape != (
    512,
    2048
):

    raise RuntimeError(
        f"Unexpected aligner W shape: {W.shape}"
    )

b = b.reshape(-1)

assert W.shape == (
    512,
    2048
)

assert b.shape == (
    512,
)

# ------------------------------------------------------------
# Construct linear aligner
# ------------------------------------------------------------

vegann_aligner = nn.Linear(
    2048,
    512,
    bias=True
)

with torch.no_grad():

    vegann_aligner.weight.copy_(
        W
    )

    vegann_aligner.bias.copy_(
        b
    )

vegann_aligner = (
    vegann_aligner
    .to(device)
)

vegann_aligner.eval()

for parameter in (
    vegann_aligner.parameters()
):

    parameter.requires_grad = False

print(
    "\n✓ VegAnn aligner loaded"
)

print("=" * 70)

LOADING VEGANN AGRICLIP ALIGNER
Checkpoint keys: ['b', 'W']
W: torch.Size([512, 2048])
b: torch.Size([512])

✓ VegAnn aligner loaded


In [55]:
# ============================================================
# CELL 14 — TEXTTOCONCEPT
# ============================================================

print("=" * 70)
print("CREATING TEXTTOCONCEPT")
print("=" * 70)

text_to_concept = TextToConcept(
    dino_model,
    "Dino"
)

print(
    "TextToConcept:",
    type(text_to_concept)
)

print(
    "Internal CLIP:",
    type(text_to_concept.clip_model)
)

print(
    "\n✓ TextToConcept ready"
)

print("=" * 70)

CREATING TEXTTOCONCEPT


TextToConcept: <class 'TextToConcept.TextToConcept'>
Internal CLIP: <class 'TextToConcept.ClipZeroShot'>

✓ TextToConcept ready


In [56]:
# ============================================================
# CELL 15 — VEGETATION PROMPTS
# ============================================================

POSITIVE_PROMPTS = [

    "vegetation",
    "green vegetation",
    "crop plants",
    "crop vegetation",
    "agricultural plants",
    "green plants",
]

NEGATIVE_PROMPTS = [

    "soil",
    "bare soil",
    "ground",
    "bare ground",
    "non-vegetated ground",
    "agricultural soil",
]

print("Prompt strategy: POSITIVE / NEGATIVE")

print("\nPositive prompts:")
for i, prompt in enumerate(POSITIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")

print("\nNegative prompts:")
for i, prompt in enumerate(NEGATIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")
    
    
    


Prompt strategy: POSITIVE / NEGATIVE

Positive prompts:
1. vegetation
2. green vegetation
3. crop plants
4. crop vegetation
5. agricultural plants
6. green plants

Negative prompts:
1. soil
2. bare soil
3. ground
4. bare ground
5. non-vegetated ground
6. agricultural soil


In [57]:

# ============================================================
# CELL 16 ENCODE POSITIVE AND NEGATIVE PROMPT SETS
# ============================================================

@torch.no_grad()
def encode_prompt_set(prompts):

    embeddings = []

    for prompt in prompts:

        embedding = text_to_concept.encode_text(
            [[prompt]]
        )

        if isinstance(embedding, np.ndarray):
            embedding = torch.from_numpy(
                embedding
            )

        if not torch.is_tensor(embedding):
            raise TypeError(
                "encode_text returned an unsupported type: "
                f"{type(embedding)}"
            )

        embedding = (
            embedding
            .detach()
            .float()
            .to(device)
            .reshape(-1)
        )

        embeddings.append(
            embedding
        )

    embeddings = torch.stack(
        embeddings,
        dim=0
    )

    # Normalize each prompt independently
    embeddings = F.normalize(
        embeddings,
        dim=1
    )

    # Average prompts
    mean_embedding = embeddings.mean(
        dim=0
    )

    # Normalize the averaged embedding
    mean_embedding = F.normalize(
        mean_embedding,
        dim=0
    )

    return embeddings, mean_embedding


# ------------------------------------------------------------
# POSITIVE
# ------------------------------------------------------------

positive_text_embeddings, positive_embedding = (
    encode_prompt_set(
        POSITIVE_PROMPTS
    )
)


# ------------------------------------------------------------
# NEGATIVE
# ------------------------------------------------------------

negative_text_embeddings, negative_embedding = (
    encode_prompt_set(
        NEGATIVE_PROMPTS
    )
)


print(
    "Positive embeddings:",
    positive_text_embeddings.shape
)

print(
    "Negative embeddings:",
    negative_text_embeddings.shape
)

print(
    "Positive mean embedding:",
    positive_embedding.shape
)

print(
    "Negative mean embedding:",
    negative_embedding.shape
)

# ============================================================
# POSITIVE - NEGATIVE CONCEPT
# ============================================================

vegetation_embedding = (
    positive_embedding
    - negative_embedding
)

vegetation_embedding = F.normalize(
    vegetation_embedding,
    dim=0
)

print(
    "Final vegetation embedding:",
    vegetation_embedding.shape
)

Positive embeddings: torch.Size([6, 512])
Negative embeddings: torch.Size([6, 512])
Positive mean embedding: torch.Size([512])
Negative mean embedding: torch.Size([512])
Final vegetation embedding: torch.Size([512])


In [58]:
# ============================================================
# CELL 17 — AGRICLIP LOCALIZATION
# ============================================================

@torch.no_grad()
def agriclip_localize(
    image,
    k=32
):

    # --------------------------------------------------------
    # DINO
    # --------------------------------------------------------

    patch_features, grid_h, grid_w = (
        extract_dino_patch_features(
            image
        )
    )

    # --------------------------------------------------------
    # Align
    # --------------------------------------------------------

    aligned_features = (
        vegann_aligner(
            patch_features.to(device)
        )
    )

    aligned_features = F.normalize(
        aligned_features,
        dim=-1
    )

    # --------------------------------------------------------
    # Text similarity
    # --------------------------------------------------------

    vegetation = F.normalize(
        vegetation_embedding.to(device),
        dim=0
    )

    scores = (
        aligned_features
        @ vegetation
    )

    # --------------------------------------------------------
    # Top-K
    # --------------------------------------------------------

    k = min(
        k,
        scores.shape[0]
    )

    top_scores, top_indices = (
        torch.topk(
            scores,
            k=k
        )
    )

    return {
        "patch_features":
            patch_features,
        "aligned_features":
            aligned_features,
        "scores":
            scores,
        "top_scores":
            top_scores.cpu().numpy(),
        "top_indices":
            top_indices.cpu().numpy(),
        "grid_h":
            grid_h,
        "grid_w":
            grid_w
    }


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_location = agriclip_localize(
    test_image,
    TOP_K
)

print("=" * 70)
print("AGRICLIP LOCALIZATION TEST")
print("=" * 70)

print(
    "Total patches:",
    len(
        test_location["scores"]
    )
)

print(
    "Selected:",
    len(
        test_location["top_indices"]
    )
)

print(
    "Top indices:",
    test_location["top_indices"]
)

print("=" * 70)

AGRICLIP LOCALIZATION TEST
Total patches: 49
Selected: 32
Top indices: [13 43  6 20 29 10  1 47 18  8 46 22 19  2  3 26 17 36 34 33 23 35 40 15
 30 12 21  7 14 41 11 28]


In [59]:
# ============================================================
# CELL 18 — AGRICLIP BOX GENERATION
# ============================================================

def agriclip_patches_to_box(
    image,
    top_indices,
    grid_h,
    grid_w,
    margin=25
):

    W, H = image.size

    patch_h = H / grid_h
    patch_w = W / grid_w

    boxes = []

    for idx in top_indices:

        row = (
            int(idx) //
            grid_w
        )

        col = (
            int(idx) %
            grid_w
        )

        x1 = int(
            col * patch_w
        )

        y1 = int(
            row * patch_h
        )

        x2 = int(
            (col + 1) * patch_w
        )

        y2 = int(
            (row + 1) * patch_h
        )

        boxes.append(
            [
                x1,
                y1,
                x2,
                y2
            ]
        )

    boxes = np.asarray(
        boxes,
        dtype=np.float32
    )

    xmin = boxes[:, 0].min()
    ymin = boxes[:, 1].min()
    xmax = boxes[:, 2].max()
    ymax = boxes[:, 3].max()

    xmin = max(
        0,
        xmin - margin
    )

    ymin = max(
        0,
        ymin - margin
    )

    xmax = min(
        W - 1,
        xmax + margin
    )

    ymax = min(
        H - 1,
        ymax + margin
    )

    return np.array(
        [
            xmin,
            ymin,
            xmax,
            ymax
        ],
        dtype=np.float32
    )


test_box = agriclip_patches_to_box(
    test_image,
    test_location["top_indices"],
    test_location["grid_h"],
    test_location["grid_w"],
    margin=BOX_MARGIN
)

print("=" * 70)
print("AGRICLIP BOX")
print("=" * 70)

print(
    "Box:",
    test_box
)

print("=" * 70)

AGRICLIP BOX
Box: [  0.   0. 511. 511.]


In [60]:
# ============================================================
# CELL 19 — LOAD ORIGINAL FINE-TUNED SAM2
# ============================================================

print("=" * 70)
print("LOADING ORIGINAL NON-AUGMENTED SAM2")
print("=" * 70)

sam2_processor = (
    Sam2Processor.from_pretrained(
        MODEL_NAME
    )
)

sam2_model = (
    Sam2Model.from_pretrained(
        MODEL_NAME
    )
)

sam2_checkpoint = torch.load(
    SAM2_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:",
    list(
        sam2_checkpoint.keys()
    )
)

if (
    "mask_decoder_state_dict"
    not in sam2_checkpoint
):

    raise RuntimeError(
        "mask_decoder_state_dict "
        "not found in SAM2 checkpoint."
    )

sam2_model.mask_decoder.load_state_dict(
    sam2_checkpoint[
        "mask_decoder_state_dict"
    ],
    strict=True
)

sam2_model = (
    sam2_model
    .to(device)
)

sam2_model.eval()

print(
    "\nCheckpoint epoch:",
    sam2_checkpoint.get(
        "epoch"
    )
)

print(
    "Best stored IoU:",
    sam2_checkpoint.get(
        "best_mean_iou"
    )
)

print(
    "\n✓ Original non-augmented SAM2 loaded"
)

print("=" * 70)

LOADING ORIGINAL NON-AUGMENTED SAM2


[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Checkpoint keys: ['epoch', 'model_name', 'mask_decoder_state_dict', 'optimizer_state_dict', 'best_mean_iou', 'history']

Checkpoint epoch: 8
Best stored IoU: 0.8724805513351916

✓ Original non-augmented SAM2 loaded


In [61]:
# ============================================================
# CELL — SET SAM2 TRAINABLE / FROZEN STATUS CORRECTLY
# ============================================================

# Freeze the entire SAM2 model first
for param in sam2_model.parameters():
    param.requires_grad = False

# Unfreeze ONLY the mask decoder
for param in sam2_model.mask_decoder.parameters():
    param.requires_grad = True

sam2_model.eval()

# ------------------------------------------------------------
# Count again
# ------------------------------------------------------------

sam2_total = sum(
    p.numel()
    for p in sam2_model.parameters()
)

sam2_trainable = sum(
    p.numel()
    for p in sam2_model.parameters()
    if p.requires_grad
)

sam2_frozen = (
    sam2_total -
    sam2_trainable
)

print("=" * 70)
print("SAM2 TRAINABLE/FROZEN STATUS")
print("=" * 70)

print(
    f"Total:    {sam2_total:,} "
    f"({sam2_total/1e6:.2f}M)"
)

print(
    f"Trainable: {sam2_trainable:,} "
    f"({sam2_trainable/1e6:.2f}M)"
)

print(
    f"Frozen:   {sam2_frozen:,} "
    f"({sam2_frozen/1e6:.2f}M)"
)

print("\nTrainable parameter groups:")

trainable_modules = {}

for name, param in sam2_model.named_parameters():

    if param.requires_grad:

        top_level = name.split(".")[0]

        trainable_modules[top_level] = (
            trainable_modules.get(
                top_level,
                0
            )
            +
            param.numel()
        )

for module_name, count in trainable_modules.items():

    print(
        f"{module_name}: "
        f"{count:,} "
        f"({count/1e6:.2f}M)"
    )

print("=" * 70)

SAM2 TRAINABLE/FROZEN STATUS
Total:    31,440,721 (31.44M)
Trainable: 4,215,109 (4.22M)
Frozen:   27,225,612 (27.23M)

Trainable parameter groups:
mask_decoder: 4,215,109 (4.22M)


In [62]:
# ============================================================
# CELL 20 — PARAMETER COUNT
# ============================================================

def count_parameters(
    model
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable


# ------------------------------------------------------------
# DINO
# ------------------------------------------------------------

dino_total, dino_trainable = (
    count_parameters(
        dino_model
    )
)

# ------------------------------------------------------------
# Aligner
# ------------------------------------------------------------

aligner_total, aligner_trainable = (
    count_parameters(
        vegann_aligner
    )
)

# ------------------------------------------------------------
# SAM2
# ------------------------------------------------------------

sam2_total, sam2_trainable = (
    count_parameters(
        sam2_model
    )
)

# ------------------------------------------------------------
# SAM2 decoder
# ------------------------------------------------------------

decoder_total, decoder_trainable = (
    count_parameters(
        sam2_model.mask_decoder
    )
)

# ------------------------------------------------------------
# Full pipeline
# ------------------------------------------------------------

full_pipeline_total = (
    dino_total
    +
    aligner_total
    +
    sam2_total
)

full_pipeline_trainable = (
    decoder_trainable
)

parameter_table = pd.DataFrame(
    [
        {
            "Component": "DINO",
            "Total Parameters": dino_total,
            "Trainable Parameters": dino_trainable
        },
        {
            "Component": "AgriCLIP Aligner",
            "Total Parameters": aligner_total,
            "Trainable Parameters": aligner_trainable
        },
        {
            "Component": "SAM2",
            "Total Parameters": sam2_total,
            "Trainable Parameters": sam2_trainable
        },
        {
            "Component": "SAM2 Mask Decoder",
            "Total Parameters": decoder_total,
            "Trainable Parameters": decoder_trainable
        },
        {
            "Component": "DINO + Aligner + SAM2",
            "Total Parameters":
                full_pipeline_total,
            "Trainable Parameters":
                full_pipeline_trainable
        }
    ]
)

print("=" * 70)
print("PARAMETER COUNTS")
print("=" * 70)

display(
    parameter_table.style.format(
        {
            "Total Parameters":
                "{:,}",
            "Trainable Parameters":
                "{:,}"
        }
    )
)

print("\nMillions:")

for row in parameter_table.itertuples():

    print(
        f"{row.Component:<30} "
        f"Total={row._2 / 1e6:.2f}M | "
        f"Trainable={row._3 / 1e6:.2f}M"
    )

PARAMETER COUNTS


,Component,Total Parameters,Trainable Parameters
0,DINO,"23,508,032",0
1,AgriCLIP Aligner,"1,049,088",0
2,SAM2,"31,440,721","4,215,109"
3,SAM2 Mask Decoder,"4,215,109","4,215,109"
4,DINO + Aligner + SAM2,"55,997,841","4,215,109"



Millions:
DINO                           Total=23.51M | Trainable=0.00M
AgriCLIP Aligner               Total=1.05M | Trainable=0.00M
SAM2                           Total=31.44M | Trainable=4.22M
SAM2 Mask Decoder              Total=4.22M | Trainable=4.22M
DINO + Aligner + SAM2          Total=56.00M | Trainable=4.22M


In [63]:
# ============================================================
# CELL 21 — INITIAL GPU MEMORY before infrence
# ============================================================

print("=" * 70)
print("GPU MEMORY STATUS")
print("=" * 70)

if torch.cuda.is_available():

    torch.cuda.synchronize()

    allocated = (
        torch.cuda.memory_allocated(0)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(0)
        / 1024**3
    )

    print(
        f"Allocated: {allocated:.4f} GB"
    )

    print(
        f"Reserved:  {reserved:.4f} GB"
    )

else:

    print(
        "CUDA unavailable."
    )

GPU MEMORY STATUS
Allocated: 0.6552 GB
Reserved:  1.2188 GB


In [64]:
# FLOPs = 2 x input x output per vector 
# 49 spatial patches 

# ============================================================
# CELL 22 — AGRICLIP ALIGNER GFLOPS
# ============================================================

NUM_PATCHES = 49
INPUT_DIM = 2048
OUTPUT_DIM = 512

aligner_flops = (
    2
    *
    NUM_PATCHES
    *
    INPUT_DIM
    *
    OUTPUT_DIM
)

aligner_gflops = (
    aligner_flops
    / 1e9
)

print("=" * 70)
print("AGRICLIP ALIGNER FLOPs")
print("=" * 70)

print(
    "Patches:",
    NUM_PATCHES
)

print(
    "Input dimension:",
    INPUT_DIM
)

print(
    "Output dimension:",
    OUTPUT_DIM
)

print(
    "Estimated FLOPs:",
    f"{aligner_flops:,}"
)

print(
    "Estimated GFLOPs:",
    f"{aligner_gflops:.6f}"
)

print("=" * 70)

AGRICLIP ALIGNER FLOPs
Patches: 49
Input dimension: 2048
Output dimension: 512
Estimated FLOPs: 102,760,448
Estimated GFLOPs: 0.102760


In [65]:
# ============================================================
# CELL 23 — PROFILER GFLOPs HELPER
# ============================================================

from torch.profiler import (
    profile,
    ProfilerActivity
)


def profile_model_gflops(
    forward_function,
    label
):

    print("\n")
    print("=" * 70)
    print(
        f"PROFILING: {label}"
    )
    print("=" * 70)

    activities = [
        ProfilerActivity.CPU
    ]

    if torch.cuda.is_available():

        activities.append(
            ProfilerActivity.CUDA
        )

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    with torch.no_grad():

        for _ in range(2):

            forward_function()

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    # --------------------------------------------------------
    # Profile
    # --------------------------------------------------------

    with profile(
        activities=activities,
        record_shapes=False,
        profile_memory=False,
        with_flops=True
    ) as prof:

        with torch.no_grad():

            forward_function()

        if torch.cuda.is_available():

            torch.cuda.synchronize()

    total_flops = 0

    for event in prof.key_averages():

        flops = getattr(
            event,
            "flops",
            0
        )

        if flops is not None:

            total_flops += flops

    gflops = (
        total_flops
        / 1e9
    )

    print(
        "Estimated FLOPs:",
        f"{total_flops:,.0f}"
    )

    print(
        "Estimated GFLOPs:",
        f"{gflops:.4f}"
    )

    print(
        "\nNote: profiler FLOPs are an estimate "
        "based on supported operators."
    )

    print("=" * 70)

    return {
        "label": label,
        "flops": total_flops,
        "gflops": gflops
    }

In [66]:
# ============================================================
# CELL 24 — PROFILE DINO GFLOPs
# ============================================================

profile_image = get_image(
    PROFILE_INDEX
)

profile_dino_input = (
    dino_transform(
        profile_image
    )
    .unsqueeze(0)
    .to(device)
)


def run_dino_profile():

    global captured_feature_map

    captured_feature_map = None

    _ = dino_model(
        profile_dino_input
    )


dino_profile = profile_model_gflops(
    run_dino_profile,
    "DINO ResNet-50"
)



PROFILING: DINO ResNet-50
Estimated FLOPs: 8,174,272,512
Estimated GFLOPs: 8.1743

Note: profiler FLOPs are an estimate based on supported operators.


USDT:2026-08-30 19:16:36 198556:198556 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-08-30 19:16:36 198556:198556 ActivityProfilerController.cpp:455] profiler_stop


In [67]:
# ============================================================
# CELL 25 — PROFILE SAM2 GFLOPs
# ============================================================

profile_loc = agriclip_localize(
    profile_image,
    TOP_K
)

profile_box = (
    agriclip_patches_to_box(
        profile_image,
        profile_loc["top_indices"],
        profile_loc["grid_h"],
        profile_loc["grid_w"],
        margin=BOX_MARGIN
    )
)

profile_inputs = sam2_processor(
    images=profile_image,
    input_boxes=[
        [
            profile_box.tolist()
        ]
    ],
    return_tensors="pt"
)

profile_inputs = {
    key: value.to(device)
    if torch.is_tensor(value)
    else value
    for key, value in profile_inputs.items()
}


def run_sam2_profile():

    _ = sam2_model(
        **profile_inputs
    )


sam2_profile = profile_model_gflops(
    run_sam2_profile,
    "Fine-tuned SAM2 + Box"
)



PROFILING: Fine-tuned SAM2 + Box


Estimated FLOPs: 210,181,796,950
Estimated GFLOPs: 210.1818

Note: profiler FLOPs are an estimate based on supported operators.


USDT:2026-08-30 19:16:36 198556:198556 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-08-30 19:16:36 198556:198556 ActivityProfilerController.cpp:455] profiler_stop


In [68]:
# ============================================================
# CELL 26 — FULL PIPELINE GFLOPs
# ============================================================

dino_gflops = (
    dino_profile["gflops"]
)

aligner_gflops = (
    aligner_gflops
)

sam2_gflops = (
    sam2_profile["gflops"]
)

full_pipeline_gflops = (
    dino_gflops
    +
    aligner_gflops
    +
    sam2_gflops
)

print("=" * 70)
print("FULL PIPELINE GFLOPs")
print("=" * 70)

print(
    f"DINO:             {dino_gflops:.4f} GFLOPs"
)

print(
    f"AgriCLIP aligner: {aligner_gflops:.6f} GFLOPs"
)

print(
    f"SAM2:             {sam2_gflops:.4f} GFLOPs"
)

print(
    "-" * 70
)

print(
    f"FULL PIPELINE:    {full_pipeline_gflops:.4f} GFLOPs"
)

print("=" * 70)

FULL PIPELINE GFLOPs
DINO:             8.1743 GFLOPs
AgriCLIP aligner: 0.102760 GFLOPs
SAM2:             210.1818 GFLOPs
----------------------------------------------------------------------
FULL PIPELINE:    218.4588 GFLOPs


In [69]:
# ============================================================
# CELL 27 — SAM2 BENCHMARK SANITY CHECK
# ============================================================

print("=" * 70)
print("SAM2 BENCHMARK SANITY CHECK")
print("=" * 70)

with torch.no_grad():

    output = sam2_model(
        **profile_inputs
    )

print(
    "Output type:",
    type(output)
)

print(
    "pred_masks:",
    output.pred_masks.shape
)

print(
    "iou_scores:",
    output.iou_scores.shape
)

print(
    "\n✓ SAM2 benchmark forward pass works"
)

print("=" * 70)

SAM2 BENCHMARK SANITY CHECK
Output type: <class 'transformers.models.sam2.modeling_sam2.Sam2ImageSegmentationOutput'>
pred_masks: torch.Size([1, 1, 3, 256, 256])
iou_scores: torch.Size([1, 1, 3])

✓ SAM2 benchmark forward pass works


In [70]:
# ============================================================
# CELL 28 — RUNTIME BENCHMARK HELPER
# ============================================================

def synchronize_cuda():

    if torch.cuda.is_available():

        torch.cuda.synchronize()


def benchmark_function(
    forward_function,
    num_images,
    warmup_images=10,
    label="Benchmark"
):

    print("=" * 70)
    print(label)
    print("=" * 70)

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    for _ in range(
        warmup_images
    ):

        forward_function()

    synchronize_cuda()

    # --------------------------------------------------------
    # Clear peak memory
    # --------------------------------------------------------

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats()

        synchronize_cuda()

    # --------------------------------------------------------
    # Timed loop
    # --------------------------------------------------------

    times = []

    for _ in tqdm(
        range(num_images),
        desc=label
    ):

        synchronize_cuda()

        start = time.perf_counter()

        forward_function()

        synchronize_cuda()

        end = time.perf_counter()

        times.append(
            end - start
        )

    times = np.asarray(
        times,
        dtype=np.float64
    )

    mean_time = (
        times.mean()
    )

    median_time = (
        np.median(times)
    )

    std_time = (
        times.std()
    )

    min_time = (
        times.min()
    )

    max_time = (
        times.max()
    )

    total_time = (
        times.sum()
    )

    images_per_sec = (
        1.0 /
        mean_time
    )

    # --------------------------------------------------------
    # Memory
    # --------------------------------------------------------

    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda.max_memory_allocated()
            /
            1024**3
        )

        peak_reserved = (
            torch.cuda.max_memory_reserved()
            /
            1024**3
        )

    else:

        peak_allocated = np.nan
        peak_reserved = np.nan

    result = {

        "label":
            label,

        "images":
            num_images,

        "mean_sec_per_image":
            mean_time,

        "median_sec_per_image":
            median_time,

        "std_sec_per_image":
            std_time,

        "min_sec_per_image":
            min_time,

        "max_sec_per_image":
            max_time,

        "images_per_sec":
            images_per_sec,

        "total_sec":
            total_time,

        "total_min":
            total_time / 60,

        "peak_gpu_allocated_gb":
            peak_allocated,

        "peak_gpu_reserved_gb":
            peak_reserved
    }

    print("\n")
    print(
        f"Mean latency: "
        f"{mean_time * 1000:.4f} ms/image"
    )

    print(
        f"Median latency: "
        f"{median_time * 1000:.4f} ms/image"
    )

    print(
        f"Std: "
        f"{std_time * 1000:.4f} ms"
    )

    print(
        f"Throughput: "
        f"{images_per_sec:.2f} images/sec"
    )

    print(
        f"Total runtime: "
        f"{total_time:.2f} sec "
        f"({total_time / 60:.2f} min)"
    )

    if torch.cuda.is_available():

        print(
            f"Peak allocated: "
            f"{peak_allocated:.4f} GB"
        )

        print(
            f"Peak reserved: "
            f"{peak_reserved:.4f} GB"
        )

    print("=" * 70)

    return result

In [71]:
# ============================================================
# CELL 29 — DINO + AGRICLIP RUNTIME
# ============================================================

benchmark_image = get_image(
    PROFILE_INDEX
)


def run_dino_agriclip():

    _ = agriclip_localize(
        benchmark_image,
        TOP_K
    )


dino_agriclip_runtime = benchmark_function(
    run_dino_agriclip,
    num_images=BENCHMARK_IMAGES,
    warmup_images=WARMUP_IMAGES,
    label="DINO + AgriCLIP Localization"
)

DINO + AgriCLIP Localization


DINO + AgriCLIP Localization:   0%|          | 0/755 [00:00<?, ?it/s]



Mean latency: 3.7931 ms/image
Median latency: 3.7556 ms/image
Std: 0.0900 ms
Throughput: 263.63 images/sec
Total runtime: 2.86 sec (0.05 min)
Peak allocated: 0.6670 GB
Peak reserved: 1.4062 GB


In [72]:
# ============================================================
# CELL 30 — SAM2 BOX SEGMENTATION RUNTIME
# ============================================================

sam2_benchmark_inputs = sam2_processor(
    images=benchmark_image,
    input_boxes=[
        [
            profile_box.tolist()
        ]
    ],
    return_tensors="pt"
)

sam2_benchmark_inputs = {
    key: value.to(device)
    if torch.is_tensor(value)
    else value
    for key, value in sam2_benchmark_inputs.items()
}


def run_sam2():

    with torch.no_grad():

        _ = sam2_model(
            **sam2_benchmark_inputs
        )


sam2_runtime = benchmark_function(
    run_sam2,
    num_images=BENCHMARK_IMAGES,
    warmup_images=WARMUP_IMAGES,
    label="Fine-tuned SAM2 + Box"
)

Fine-tuned SAM2 + Box


Fine-tuned SAM2 + Box:   0%|          | 0/755 [00:00<?, ?it/s]



Mean latency: 19.2677 ms/image
Median latency: 19.2036 ms/image
Std: 0.1324 ms
Throughput: 51.90 images/sec
Total runtime: 14.55 sec (0.24 min)
Peak allocated: 0.9414 GB
Peak reserved: 1.4062 GB


In [73]:
# ============================================================
# CELL 31 — FULL AGRICLIP → SAM2 PIPELINE RUNTIME
# ============================================================

def run_full_pipeline():

    # --------------------------------------------------------
    # 1. AgriCLIP localization
    # --------------------------------------------------------

    localization = (
        agriclip_localize(
            benchmark_image,
            TOP_K
        )
    )

    # --------------------------------------------------------
    # 2. AgriCLIP box
    # --------------------------------------------------------

    box = (
        agriclip_patches_to_box(
            benchmark_image,
            localization["top_indices"],
            localization["grid_h"],
            localization["grid_w"],
            margin=BOX_MARGIN
        )
    )

    # --------------------------------------------------------
    # 3. SAM2 processor
    # --------------------------------------------------------

    inputs = sam2_processor(
        images=benchmark_image,
        input_boxes=[
            [
                box.tolist()
            ]
        ],
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # 4. SAM2
    # --------------------------------------------------------

    with torch.no_grad():

        _ = sam2_model(
            **inputs
        )


full_pipeline_runtime = benchmark_function(
    run_full_pipeline,
    num_images=BENCHMARK_IMAGES,
    warmup_images=WARMUP_IMAGES,
    label="FULL AgriCLIP → SAM2"
)

FULL AgriCLIP → SAM2


FULL AgriCLIP → SAM2:   0%|          | 0/755 [00:00<?, ?it/s]



Mean latency: 28.1804 ms/image
Median latency: 28.1382 ms/image
Std: 0.3837 ms
Throughput: 35.49 images/sec
Total runtime: 21.28 sec (0.35 min)
Peak allocated: 0.9536 GB
Peak reserved: 1.4062 GB


In [74]:
# ============================================================
# CELL 32 — RUNTIME SUMMARY
# ============================================================

print("=" * 70)
print("RUNTIME SUMMARY")
print("=" * 70)

runtime_components = pd.DataFrame(
    [
        dino_agriclip_runtime,
        sam2_runtime,
        full_pipeline_runtime
    ]
)

display(
    runtime_components.style.format(
        {
            "mean_sec_per_image":
                "{:.6f}",

            "median_sec_per_image":
                "{:.6f}",

            "std_sec_per_image":
                "{:.6f}",

            "min_sec_per_image":
                "{:.6f}",

            "max_sec_per_image":
                "{:.6f}",

            "images_per_sec":
                "{:.2f}",

            "total_sec":
                "{:.2f}",

            "total_min":
                "{:.2f}",

            "peak_gpu_allocated_gb":
                "{:.4f}",

            "peak_gpu_reserved_gb":
                "{:.4f}"
        }
    )
)

RUNTIME SUMMARY


,label,images,mean_sec_per_image,median_sec_per_image,std_sec_per_image,min_sec_per_image,max_sec_per_image,images_per_sec,total_sec,total_min,peak_gpu_allocated_gb,peak_gpu_reserved_gb
0,DINO + AgriCLIP Localization,755,0.003793,0.003756,0.000090,0.003680,0.004501,263.63,2.86,0.05,0.6670,1.4062
1,Fine-tuned SAM2 + Box,755,0.019268,0.019204,0.000132,0.019005,0.019806,51.90,14.55,0.24,0.9414,1.4062
2,FULL AgriCLIP → SAM2,755,0.028180,0.028138,0.000384,0.027850,0.034594,35.49,21.28,0.35,0.9536,1.4062


In [75]:
# ============================================================
# CELL 33 — AGRICLIP OVERHEAD ANALYSIS
# ============================================================

localization_ms = (
    dino_agriclip_runtime[
        "mean_sec_per_image"
    ]
    * 1000
)

sam2_ms = (
    sam2_runtime[
        "mean_sec_per_image"
    ]
    * 1000
)

full_ms = (
    full_pipeline_runtime[
        "mean_sec_per_image"
    ]
    * 1000
)

agriclip_overhead_percent = (
    localization_ms
    /
    sam2_ms
    *
    100
)

extra_full_vs_sam2 = (
    full_ms
    -
    sam2_ms
)

print("=" * 70)
print("COMPUTATIONAL OVERHEAD")
print("=" * 70)

print(
    f"DINO + AgriCLIP: "
    f"{localization_ms:.4f} ms/image"
)

print(
    f"SAM2: "
    f"{sam2_ms:.4f} ms/image"
)

print(
    f"Full pipeline: "
    f"{full_ms:.4f} ms/image"
)

print(
    f"\nAgriCLIP localization "
    f"relative to SAM2: "
    f"{agriclip_overhead_percent:.2f}%"
)

print(
    f"Additional full-pipeline "
    f"time over SAM2: "
    f"{extra_full_vs_sam2:.4f} ms/image"
)

print("=" * 70)

COMPUTATIONAL OVERHEAD
DINO + AgriCLIP: 3.7931 ms/image
SAM2: 19.2677 ms/image
Full pipeline: 28.1804 ms/image

AgriCLIP localization relative to SAM2: 19.69%
Additional full-pipeline time over SAM2: 8.9128 ms/image


In [76]:
# ============================================================
# CELL 34 — 755 IMAGE COMPUTATIONAL COST
# ============================================================

NUM_IMAGES = 755

dino_agriclip_total_755 = (
    dino_agriclip_runtime[
        "mean_sec_per_image"
    ]
    *
    NUM_IMAGES
)

sam2_total_755 = (
    sam2_runtime[
        "mean_sec_per_image"
    ]
    *
    NUM_IMAGES
)

full_total_755 = (
    full_pipeline_runtime[
        "mean_sec_per_image"
    ]
    *
    NUM_IMAGES
)

cost_table = pd.DataFrame(
    [
        {
            "Method":
                "DINO + AgriCLIP",
            "Images":
                NUM_IMAGES,
            "Mean ms/image":
                localization_ms,
            "Images/sec":
                dino_agriclip_runtime[
                    "images_per_sec"
                ],
            "Estimated 755-image sec":
                dino_agriclip_total_755,
            "Estimated minutes":
                dino_agriclip_total_755 / 60
        },

        {
            "Method":
                "SAM2 + Box",
            "Images":
                NUM_IMAGES,
            "Mean ms/image":
                sam2_ms,
            "Images/sec":
                sam2_runtime[
                    "images_per_sec"
                ],
            "Estimated 755-image sec":
                sam2_total_755,
            "Estimated minutes":
                sam2_total_755 / 60
        },

        {
            "Method":
                "FULL AgriCLIP → SAM2",
            "Images":
                NUM_IMAGES,
            "Mean ms/image":
                full_ms,
            "Images/sec":
                full_pipeline_runtime[
                    "images_per_sec"
                ],
            "Estimated 755-image sec":
                full_total_755,
            "Estimated minutes":
                full_total_755 / 60
        }
    ]
)

print("=" * 70)
print("755-IMAGE COMPUTATIONAL COST")
print("=" * 70)

display(
    cost_table.style.format(
        {
            "Mean ms/image":
                "{:.4f}",

            "Images/sec":
                "{:.2f}",

            "Estimated 755-image sec":
                "{:.2f}",

            "Estimated minutes":
                "{:.2f}"
        }
    )
)

755-IMAGE COMPUTATIONAL COST


,Method,Images,Mean ms/image,Images/sec,Estimated 755-image sec,Estimated minutes
0,DINO + AgriCLIP,755,3.7931,263.63,2.86,0.05
1,SAM2 + Box,755,19.2677,51.90,14.55,0.24
2,FULL AgriCLIP → SAM2,755,28.1804,35.49,21.28,0.35


In [77]:
# ============================================================
# CELL 35 — FULL PIPELINE GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    synchronize_cuda()

    run_full_pipeline()

    synchronize_cuda()

    peak_allocated = (
        torch.cuda.max_memory_allocated()
        /
        1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved()
        /
        1024**3
    )

    print("=" * 70)
    print("FULL PIPELINE GPU MEMORY")
    print("=" * 70)

    print(
        f"Peak allocated: "
        f"{peak_allocated:.4f} GB"
    )

    print(
        f"Peak reserved: "
        f"{peak_reserved:.4f} GB"
    )

    print("=" * 70)

else:

    print(
        "CUDA unavailable."
    )

FULL PIPELINE GPU MEMORY
Peak allocated: 0.9536 GB
Peak reserved: 1.2266 GB


In [78]:
# ============================================================
# CELL 36 — FINAL COMPUTATIONAL SUMMARY
# ============================================================

summary_rows = [

    {
        "Component":
            "DINO",
        "Parameters_M":
            dino_total / 1e6,
        "Trainable_M":
            dino_trainable / 1e6,
        "GFLOPs":
            dino_gflops,
        "Mean_ms":
            np.nan,
        "Images_per_sec":
            np.nan
    },

    {
        "Component":
            "AgriCLIP Aligner",
        "Parameters_M":
            aligner_total / 1e6,
        "Trainable_M":
            aligner_trainable / 1e6,
        "GFLOPs":
            aligner_gflops,
        "Mean_ms":
            np.nan,
        "Images_per_sec":
            np.nan
    },

    {
        "Component":
            "SAM2",
        "Parameters_M":
            sam2_total / 1e6,
        "Trainable_M":
            sam2_trainable / 1e6,
        "GFLOPs":
            sam2_gflops,
        "Mean_ms":
            sam2_ms,
        "Images_per_sec":
            sam2_runtime[
                "images_per_sec"
            ]
    },

    {
        "Component":
            "DINO + AgriCLIP",
        "Parameters_M":
            (
                dino_total +
                aligner_total
            ) / 1e6,
        "Trainable_M":
            0.0,
        "GFLOPs":
            dino_gflops +
            aligner_gflops,
        "Mean_ms":
            localization_ms,
        "Images_per_sec":
            dino_agriclip_runtime[
                "images_per_sec"
            ]
    },

    {
        "Component":
            "FULL AgriCLIP → SAM2",
        "Parameters_M":
            full_pipeline_total / 1e6,
        "Trainable_M":
            full_pipeline_trainable / 1e6,
        "GFLOPs":
            full_pipeline_gflops,
        "Mean_ms":
            full_ms,
        "Images_per_sec":
            full_pipeline_runtime[
                "images_per_sec"
            ]
    }
]

computational_summary_df = pd.DataFrame(
    summary_rows
)

print("=" * 70)
print("FINAL COMPUTATIONAL SUMMARY")
print("=" * 70)

display(
    computational_summary_df.style.format(
        {
            "Parameters_M":
                "{:.2f}",

            "Trainable_M":
                "{:.2f}",

            "GFLOPs":
                "{:.4f}",

            "Mean_ms":
                "{:.4f}",

            "Images_per_sec":
                "{:.2f}"
        }
    )
)

FINAL COMPUTATIONAL SUMMARY


,Component,Parameters_M,Trainable_M,GFLOPs,Mean_ms,Images_per_sec
0,DINO,23.51,0.00,8.1743,nan,nan
1,AgriCLIP Aligner,1.05,0.00,0.1028,nan,nan
2,SAM2,31.44,4.22,210.1818,19.2677,51.90
3,DINO + AgriCLIP,24.56,0.00,8.2770,3.7931,263.63
4,FULL AgriCLIP → SAM2,56.00,4.22,218.4588,28.1804,35.49


In [79]:
# ============================================================
# VERIFY ACTUAL TRAINABLE SAM2 PARAMETERS
# ============================================================

sam2_trainable = []
sam2_frozen = []

for name, param in sam2_model.named_parameters():

    if param.requires_grad:
        sam2_trainable.append(
            (name, param.numel())
        )
    else:
        sam2_frozen.append(
            (name, param.numel())
        )

trainable_count = sum(
    n for _, n in sam2_trainable
)

total_count = sum(
    p.numel()
    for p in sam2_model.parameters()
)

print("=" * 70)
print("SAM2 PARAMETER STATUS")
print("=" * 70)

print(
    f"Total parameters: "
    f"{total_count:,} "
    f"({total_count / 1e6:.2f}M)"
)

print(
    f"Trainable parameters: "
    f"{trainable_count:,} "
    f"({trainable_count / 1e6:.2f}M)"
)

print(
    f"Frozen parameters: "
    f"{total_count - trainable_count:,} "
    f"({(total_count-trainable_count)/1e6:.2f}M)"
)

print("\nTrainable modules:")

for name, count in sam2_trainable:
    print(
        f"{name}: {count:,}"
    )

print("=" * 70)

SAM2 PARAMETER STATUS
Total parameters: 31,440,721 (31.44M)
Trainable parameters: 4,215,109 (4.22M)
Frozen parameters: 27,225,612 (27.23M)

Trainable modules:
mask_decoder.iou_token.weight: 256
mask_decoder.mask_tokens.weight: 1,024
mask_decoder.transformer.layers.0.self_attn.q_proj.weight: 65,536
mask_decoder.transformer.layers.0.self_attn.q_proj.bias: 256
mask_decoder.transformer.layers.0.self_attn.k_proj.weight: 65,536
mask_decoder.transformer.layers.0.self_attn.k_proj.bias: 256
mask_decoder.transformer.layers.0.self_attn.v_proj.weight: 65,536
mask_decoder.transformer.layers.0.self_attn.v_proj.bias: 256
mask_decoder.transformer.layers.0.self_attn.o_proj.weight: 65,536
mask_decoder.transformer.layers.0.self_attn.o_proj.bias: 256
mask_decoder.transformer.layers.0.layer_norm1.weight: 256
mask_decoder.transformer.layers.0.layer_norm1.bias: 256
mask_decoder.transformer.layers.0.cross_attn_token_to_image.q_proj.weight: 32,768
mask_decoder.transformer.layers.0.cross_attn_token_to_image.q_p

In [80]:
# ============================================================
# CELL 37 — SAVE COMPUTATIONAL RESULTS
# ============================================================

parameter_table.to_csv(
    RESULTS_DIR /
    "parameter_counts.csv",
    index=False
)

runtime_components.to_csv(
    RESULTS_DIR /
    "runtime_components.csv",
    index=False
)

cost_table.to_csv(
    RESULTS_DIR /
    "755_image_cost.csv",
    index=False
)

computational_summary_df.to_csv(
    RESULTS_DIR /
    "computational_summary.csv",
    index=False
)

print("=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(
    "Directory:",
    RESULTS_DIR
)

print(
    "\nFiles:"
)

print(
    " - parameter_counts.csv"
)

print(
    " - runtime_components.csv"
)

print(
    " - 755_image_cost.csv"
)

print(
    " - computational_summary.csv"
)

print("=" * 70)

RESULTS SAVED
Directory: /home/zahranadeem/thesis/results/computational_cost

Files:
 - parameter_counts.csv
 - runtime_components.csv
 - 755_image_cost.csv
 - computational_summary.csv


In [81]:
# ============================================================
# CELL 38 — THESIS SUMMARY TABLE
# ============================================================

thesis_table = pd.DataFrame(
    [
        {
            "Method":
                "Fine-tuned SAM2 + Box",
            "Prompt":
                "Box",
            "Parameters (M)":
                sam2_total / 1e6,
            "GFLOPs":
                sam2_gflops,
            "Latency (ms/image)":
                sam2_ms,
            "Throughput (images/s)":
                sam2_runtime[
                    "images_per_sec"
                ],
            "Peak GPU (GB)":
                sam2_runtime[
                    "peak_gpu_reserved_gb"
                ]
        },

        {
            "Method":
                "AgriCLIP + SAM2",
            "Prompt":
                "AgriCLIP K=32 box",
            "Parameters (M)":
                full_pipeline_total / 1e6,
            "GFLOPs":
                full_pipeline_gflops,
            "Latency (ms/image)":
                full_ms,
            "Throughput (images/s)":
                full_pipeline_runtime[
                    "images_per_sec"
                ],
            "Peak GPU (GB)":
                full_pipeline_runtime[
                    "peak_gpu_reserved_gb"
                ]
        }
    ]
)

print("=" * 70)
print("THESIS-READY COMPUTATIONAL TABLE")
print("=" * 70)

display(
    thesis_table.style.format(
        {
            "Parameters (M)":
                "{:.2f}",

            "GFLOPs":
                "{:.4f}",

            "Latency (ms/image)":
                "{:.4f}",

            "Throughput (images/s)":
                "{:.2f}",

            "Peak GPU (GB)":
                "{:.4f}"
        }
    )
)

thesis_table.to_csv(
    RESULTS_DIR /
    "thesis_computational_table.csv",
    index=False
)

print(
    "\n✓ Thesis computational table saved"
)

THESIS-READY COMPUTATIONAL TABLE


,Method,Prompt,Parameters (M),GFLOPs,Latency (ms/image),Throughput (images/s),Peak GPU (GB)
0,Fine-tuned SAM2 + Box,Box,31.44,210.1818,19.2677,51.90,1.4062
1,AgriCLIP + SAM2,AgriCLIP K=32 box,56.00,218.4588,28.1804,35.49,1.4062



✓ Thesis computational table saved
